# 5. Partitions and groups

Queries answer "which compartments match this?". Aggregation asks the opposite
question: "for each stratum, which compartments belong to it?". summer4 provides
two methods for that, and the difference between them is how they treat ragged
holes.

In [ ]:
from summer4 import Property, PropertyMap

state = Property("state", ("S", "I", "R"))
age = Property("age", ("0-4", "5-9", "10+"))
severity = Property("severity", ("mild", "severe"))

pmap = (
    PropertyMap.from_property(state)
    .stratify(age)
    .stratify(severity, where=state["I"])
)
assert pmap.size == 12

## `partition` — one property, every trait

`partition(prop)` returns a dictionary from `Trait` to index array, with one
entry per trait **whether or not it is populated**. Together the entries cover
exactly `prop.present()`.

In [ ]:
by_age = pmap.partition(age)

assert [trait.name for trait in by_age] == ["0-4", "5-9", "10+"]
assert sum(indices.size for indices in by_age.values()) == pmap.size

for trait, indices in by_age.items():
    print(f"{trait.name:>5}  {indices.tolist()}")

Age applies to every compartment, so its partition covers the whole table.
Severity does not, so its partition covers only the infectious rows — the
difference is precisely the absent set.

In [ ]:
by_severity = pmap.partition(severity)
covered = sum(indices.size for indices in by_severity.values())

assert covered == pmap.select(severity.present()).size == 6
assert covered + pmap.select(severity.absent()).size == pmap.size

for trait, indices in by_severity.items():
    print(f"{trait.name:>7}  {indices.tolist()}")

`partition` accepts a property **name** as well as an object, which is
convenient when working from a map alone.

In [ ]:
assert {t.name: i.tolist() for t, i in pmap.partition("age").items()} == {
    t.name: i.tolist() for t, i in by_age.items()
}

## `group_by` — several properties, only realised combinations

`group_by(*props)` yields `(traits, indices)` for every combination that
actually exists. Combinations with no compartments are not yielded, and any
compartment missing one of the requested properties is skipped entirely.

In [ ]:
for traits, indices in pmap.group_by(state, age):
    names = "/".join(trait.name for trait in traits)
    print(f"{names:<10} {indices.tolist()}")

Groups come out ordered lexicographically by **trait code** — that is, by
declaration order of the properties' traits, not alphabetically. `S` precedes
`I` precedes `R` because that is how `state` was declared.

In [ ]:
order = [tuple(t.name for t in traits) for traits, _ in pmap.group_by(state, age)]
assert order[:4] == [("S", "0-4"), ("S", "5-9"), ("S", "10+"), ("I", "0-4")]

### Ragged properties drop rows

Grouping by a ragged property silently restricts the result to compartments that
have it. This is the behaviour you want for "incidence by state and severity",
and the behaviour you must *not* rely on for a population total.

In [ ]:
groups = list(pmap.group_by(state, severity))
covered = sum(indices.size for _, indices in groups)

assert covered == 6, "only the infectious compartments have a severity"
assert [tuple(t.name for t in traits) for traits, _ in groups] == [
    ("I", "mild"),
    ("I", "severe"),
]

```{admonition} Totals and ragged groups
:class: warning

`sum(indices.size for _, indices in pmap.group_by(...))` equals `pmap.size` only
when every requested property is present on every compartment. Assert it when
you depend on it.
```

In [ ]:
def covers_everything(pm, *props):
    total = sum(indices.size for _, indices in pm.group_by(*props))
    return total == pm.size


assert covers_everything(pmap, state, age)
assert not covers_everything(pmap, state, severity)

## Aggregating a vector over strata

Index arrays are ordinary NumPy indices, so aggregation is a one-liner. This is
how a derived output such as "prevalence by age band" will be computed once the
solver layer exists.

In [ ]:
import numpy as np

# Pretend this came out of an integrator.
population = np.linspace(100.0, 1200.0, pmap.size)

by_age_total = {
    trait.name: float(population[indices].sum()) for trait, indices in pmap.partition(age).items()
}
assert abs(sum(by_age_total.values()) - population.sum()) < 1e-9

for name, value in by_age_total.items():
    print(f"{name:>5}  {value:10.1f}")

In [ ]:
infectious_by_severity = {
    "/".join(t.name for t in traits): float(population[indices].sum())
    for traits, indices in pmap.group_by(state, severity)
}
print(infectious_by_severity)

# The infectious total is the sum of its severity groups, because severity is
# present on exactly the infectious compartments.
assert abs(
    sum(infectious_by_severity.values()) - population[pmap.select(state["I"])].sum()
) < 1e-9

---

Next: {doc}`06-immutability-and-provenance` covers what a map remembers about
how it was built.